In [0]:
#Imports necessários para executar o notebook
from pyspark.sql.functions import when, col, coalesce, to_date, date_format,

Lendo os arquivos

In [0]:
df_catalogo_filmes = spark.read.csv("/Volumes/workspace/default/dados_streaming/catalogo_filmes.csv",
                                     header = True, #Primeira linha do csv é de cabeçalho
                                    inferSchema = True) #O Spark supõe o tipo do dado apresentado

display (df_catalogo_filmes) #Podemos ver como ficou a tabela e seu schema

In [0]:
df_usuarios = spark.read.csv("/Volumes/workspace/default/dados_streaming/usuarios.csv",
                             header = True,
                             inferSchema = True)

display(df_usuarios)

In [0]:
df_logs_streaming = spark.read.csv("/Volumes/workspace/default/dados_streaming/logs_streaming.csv",
                                   header = True,
                                   inferSchema = True)

display(df_logs_streaming)

Salvando os arquivos lidos na camada bronze

In [0]:
#Salvando os cvs importados no Databricks na camada bronze, o modo overwrite salva a tabela de um modo que se eu rodar o código novamente, a tabela atual será sobrescrevida, não acumulando histórico.
df_catalogo_filmes.write.mode("overwrite").saveAsTable("workspace.default.catalago_filmes_bronze")
df_usuarios.write.mode("overwrite").saveAsTable("workspace.default.usuarios_bronze")
df_logs_streaming.write.mode("overwrite").saveAsTable("workspace.default.logs_streaming_bronze")

Renomeando as colunas dos dfs

In [0]:
#Renomeando os nomes das colunas de inglês para português do df_catalogo_filmes
#Criando um dicionário com todas colunas que quero traduzir 
traducao_df_catalogo_filmes = {
    "movie_id":"id_filme",
    "title": "título",
    "genre": "gênero", 
    "release_year": "ano_lançamento",
    "duration_minutes": "duração_minutos",
    "content_type": "tipo",
    "original_language": "língua_original"}

#Usando a lista para fazer a tradução das colunas em lote
df_catalogo_filmes = df_catalogo_filmes.select([
    df_catalogo_filmes[col].alias(traducao_df_catalogo_filmes.get(col, col)) 
    for col in df_catalogo_filmes.columns])

#Renomeando os nomes das colunas de inglês para português do df_usuarios
traducao_df_usuarios = {
"user_id": "id_usuário",
"user_name": "nome_usuario",
"age": "idade",
"country": "país",
"signup_date": "data_assinatura",
"subscription_type": "tipo_assinatura"}

df_usuarios = df_usuarios.select([
    df_usuarios[col].alias(traducao_df_usuarios.get(col, col)) 
    for col in df_usuarios.columns])

#Renomeando os nomes das colunas de inglês para português do df_logs_streaming
traducao_df_logs_streaming = {
"session_id": "id_seção",
"user_id": "id_usuário",
"movie_id": "id_filme",
"device": "dispositivo",
"country": "país",
"subscription_type": "tipo_assinatura",
"watch_time_minutes": "tempo_assistido",
"watch_date": "dia",
"rating": "nota",
"playback_status": "playback_status"}

df_logs_streaming = df_logs_streaming.select([
    df_logs_streaming[col].alias(traducao_df_logs_streaming.get(col, col)) 
    for col in df_logs_streaming.columns])


Tratando os dados do df_catalago_filmes

In [0]:
#Traduzindo os valores da coluna gênero onde está escrito em inglês para português
df_catalogo_filmes = df_catalogo_filmes.withColumn(
    "gênero",
    when(col("gênero") == "Action", "Ação")
    .when(col("gênero") == "Documentary", "Documentário")
    .when(col("gênero") == "Animation", "Animação")
    .when(col("gênero") == "Horror", "Terror")
    .when(col("gênero") == "Comedy", "Comédia")
    .when(col("gênero") == "Sci-Fi", "Ficção Científica")
    .otherwise(col("gênero"))
)

In [0]:
#Padronizando os valores da coluna tipo para Filme, Série e Minissérie
df_catalogo_filmes = df_catalogo_filmes.withColumn(
    "tipo",
    when(col("tipo").isin("Movie", "movie"), "Filme")
    .when(col("tipo").isin("SERIES", "series"), "Série")
    .when(col("tipo").isin("miniseries", "Miniseries"), "Minissérie")
    .otherwise(col("tipo"))
)

In [0]:
#Traduzindo e padronizando os valores da coluna língua_original para português
df_catalogo_filmes = df_catalogo_filmes.withColumn(
    "língua_original",
    when(col("língua_original").isin("en", "EN-US"), "Inglês")
    .when(col("língua_original").isin("es"), "Espanhol")
    .when(col("língua_original").isin("pt", "pt-BR"), "Português")
    .otherwise(col("língua_original"))
)

In [0]:
#Resultado do tratamendo de dados do df_catalogo_filmes
display(df_catalogo_filmes)

In [0]:
#Salvando o df_catalago_filmes tratado na camada prata (pronta para uso)
df_catalogo_filmes.write.mode("overwrite").saveAsTable("workspace.default.catalago_filmes_prata")

Tratando os dados do df_usuarios

In [0]:
display(df_usuarios)

In [0]:
#Traduzindo e padronizando os valores da coluna país do usuário

df_usuarios = df_usuarios.withColumn(
    "país",
    when(col("país") == "USA", "Estados Unidos")
    .when(col("país") == "United States", "Estados Unidos")
    .when(col("país") == "Brazil", "Brasil")
    .when(col("país") == "BR", "Brasil")
    .when(col("país") == "Colombia", "Colômbia")
    .when(col("país") == "Spain", "Espanha")
    .when(col("país").isin("Mexico", "México"), "México")
    .otherwise(col("país"))
)

In [0]:
#Padronizando a coluna data_assinatura para o padrão brasileiro de datas (dd/MM/yyyy)
df_usuarios = df_usuarios.withColumn(
    "data_assinatura",
    date_format(
        coalesce(
            to_date(col("data_assinatura"), "yyyy-MM-dd"),
            to_date(col("data_assinatura"), "dd/MM/yyyy"),
            to_date(col("data_assinatura"), "MM/dd/yyyy"),
            to_date(col("data_assinatura"), "yyyy/MM/dd"),
            to_date(col("data_assinatura"), "dd-MM-yyyy"),
            to_date(col("data_assinatura"), "yyyyMMdd"),
        ),
        "dd/MM/yyyy"
    )
)

In [0]:
#Traduzindo e padronizando os valores da coluna tipo_assinatura do usuário
df_usuarios = df_usuarios.withColumn(
    "tipo_assinatura",
    when(col("tipo_assinatura") == "Free", "Grátis")
    .when(col("tipo_assinatura").isin("Basic", "basic"), "Básico")
    .when(col("tipo_assinatura") == "premium", "Premium")
    .when(col("tipo_assinatura") == "standard", "Padrão")
    .when(col("tipo_assinatura") == "Trial", "Teste Gratuito")
    .otherwise(col("tipo_assinatura"))
)


In [0]:
#Salvando o df_usuarios tratado na camada bronze
df_usuarios.write.mode("overwrite").saveAsTable("workspace.default.usuarios_prata")

In [0]:
%sql
select distinct tipo_assinatura
 from workspace.default.usuarios_bronze

In [0]:

df_usuarios
ver distinct 

country
subscription_type
signup_date

In [0]:
tabela: df_logs_streaming

ver distinct 

subscription_type

playback_status


identificar distintos e ver onde desejo substituir 


considerar so numeros rating


Renomeando colunas

In [0]:
#Tratando a coluna watch_time_minutes do df_logs_streaming
from pyspark.sql.functions import col #importando a função coluna

df_logs_streaming = spark.read.table("workspace.default.logs_streaming_bronze")

df_logs_streaming = df_logs_streaming.withColumn( #Cria ou substitui uma coluna 
    "watch_time_minutes",
    col("watch_time_minutes").cast("integer") #Muda o tipo da coluna
)

In [0]:
#Tratando a coluna playback_status

from pyspark.sql.functions import lower

df_logs_streaming = df_logs_streaming.withColumn(
    "playback_status",
    lower(col("playback_status")) #Transforma todas letras dos campos dessa coluna em letras minúsculas 
)

In [0]:
#Criando coluna watch_time_minutes de minutos em horas

df_logs_streaming = df_logs_streaming.withColumn(
    "watch_time_hours",
    (col("watch_time_minutes"))/60 #Divindo por 60 transformamos minutos em horas 
)

In [0]:
#Classificando quando o filme é um filme longo ou curto

from pyspark.sql.functions import when

df_logs_streaming = df_logs_streaming.withColumn(
    "watch_categories",
    when(
        col("watch_time_minutes") >= 120, "Longo")
    .otherwise("Curto")
    )